# result comparison of MCMC vs FM
This notebook is to compare the inferred posterior (conditioned on the same simulated lightcurve) resulting from MCMC sampling and a trained flow matching model 

In [ ]:
import os
import sys

from corner import corner

sys.path.append('..')

import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml

from src.c2st import c2st
from src.flow_matching.helpers import choose_device, count_model_parameters
from src.flow_matching.loader import (
    load_model_info,
    load_trained_model,
    prob_path_from_config,
)
from src.flow_matching.neural_posterior import NeuralPosterior
from src.flow_matching.simulator import Model as BatchedModel
from src.helpers import gen_parameter_labels, plot_posterior_samples
from src.simulator import Model

device = choose_device()

# Loading in the FM model

In [ ]:
# fill in desired job_id or directory name 
job_id = "15101905"#"15102039" #False

# loading the model (found via job_id, or path)
vector_field, classifier, config = load_trained_model(job_id, device, ema=False)

transdimensional = not config['training']['fixed_N']
path = prob_path_from_config(config)
model_info = load_model_info(config, path=path, device=device)

In [ ]:
inf_params  = model_info['inf_params']
N           = model_info['N']
vector_dim  = model_info['vector_dim']
burstparams = model_info['burstparams']
mean        = model_info['mean']
std         = model_info['std']
YBKG        = config['path']['p_data']['init_params']['model_params']['ybkg']

In [ ]:
count_model_parameters(vector_field.component_classifier)

# Loading the matching MCMC samples

In [ ]:
# check if run with same N, inf_params and burstparams exists
def mcmc_settings_exist(settings, dictionary):
    print(settings)

    for key, value in dictionary.items():

        # key does not exits
        if not settings.get(key, False):
            return False
        if key == 'burstparams':
            same_params = compare_dicts(ignore_keys=settings['inf_params'], d1=settings[key], d2=value)
        elif key == 'inf_params':
            if not set(settings[key]) == set(value):
                return False
        elif key == 'N':
            if settings[key] != value:
                return False
        
    return True 

def compare_dicts(ignore_keys, d1, d2, rel_tol=1e-9, abs_tol=0.0):
    if d1.keys() != d2.keys():
        return False
    return np.all(np.isclose(d1[k], d2[k], rel_tol=rel_tol, abs_tol=abs_tol) for k in d1 if k not in ignore_keys)

def mcmc_run_exists(N, inf_params, burstparams, save_dir='../MCMC_runs'):
    dirs = os.listdir(save_dir)

    for dir in dirs:

        with open(os.path.join(save_dir, dir, 'settings.yaml'), 'r') as f:
            settings = yaml.safe_load(f)

        if mcmc_settings_exist(settings, {'N':N, 'inf_params':inf_params, 'burstparams':burstparams}):
            return os.path.join(save_dir, dir)
        
    return False

N_true = 3
run_path = "/home/nvandermeulen/github/FRB-flow/MCMC_runs/run_06-10_14:03:18"#mcmc_run_exists(N_true, inf_params, burstparams)
if not run_path:
    print("No matching MCMC run found")
else:
    print(f'matching mcmc run found at {run_path}!')
    settings_file = open(os.path.join(run_path, 'settings.yaml'), 'r')
    burstparams = yaml.safe_load(settings_file)['burstparams']
    settings_file.close()

    samples = np.load(os.path.join(run_path, 'samples.npy'))
    simulated_counts = np.load(os.path.join(run_path, 'simulated_counts.npy'))

In [ ]:
import pickle
with open(run_path + '/sampler.pkl', "rb") as f:
    sampler = pickle.load(f)
sampler

In [ ]:
print("Autocorrelation time: ", sampler.get_autocorr_time())
max_tau = max(sampler.get_autocorr_time())
print(max_tau)
print('effective sample size: ', len(samples) / 20 / max_tau )

In [ ]:
samples.shape

In [ ]:
plt.plot(simulated_counts)

# Sample the t-FMPE posterior 

In [ ]:
learned_posterior = NeuralPosterior(
    vector_field,
    classifier,
    model_info,
    path,
    device=device
)

In [ ]:
num_samples = 2_500
samples_per_batch = 2_500

Ns, theta_samples = learned_posterior.sample(num_samples, simulated_counts, samples_per_batch, time_steps=200)
p_N = learned_posterior.p_N

In [ ]:
Ns = Ns.to('cpu')
theta_samples

In [ ]:
plt.bar(range(1, len(p_N.cpu().detach().numpy())+1), height=p_N.cpu().detach().numpy())
plt.xlabel("$N_{pred}$")
plt.title('p(N|y)')

# Corner plot overlay

In [ ]:
# --- for when you pre-load the FM samples --- 
FM_samples = np.load("FM_corner_FRB_115B.npy")
range_ = [1, 1]#[0.95, 1]
N_inf = 3  # choose for which N to make corner plot (for FM)
N_inf = N if not transdimensional else N_inf

samples_copy = samples.copy()

# make vector correct size 
# (f.e. if N=2 select t0_1, t0_2, rise_1, rise_2 from vector structured like [t0_1 ... t0_Nmax,  rise_1 ... rise_Nmax])
bs, dim = FM_samples.shape
new_dim = N_inf * len(inf_params) 

data = [FM_samples, samples_copy]

true_values = np.array([burstparams[key][:N_inf] for key in inf_params]).flatten()

colors=['blue', 'red']
labels=['FM', 'MCMC']

In [ ]:
range_ = [1, 1]#[0.95, 1]
N_inf = 3  # choose for which N to make corner plot (for FM)
N_inf = N if not transdimensional else N_inf
FM_samples = theta_samples[(Ns == N_inf).view(Ns.size(0))]# samples where N == N_inf

samples_copy = samples.copy()

# make vector correct size 
# (f.e. if N=2 select t0_1, t0_2, rise_1, rise_2 from vector structured like [t0_1 ... t0_Nmax,  rise_1 ... rise_Nmax])
bs, dim = FM_samples.shape
new_dim = N_inf * len(inf_params)
FM_samples = FM_samples.view(-1, len(inf_params), N)[:, :, :N_inf].reshape(bs, new_dim)  

data = [FM_samples.cpu().numpy(), samples_copy]

true_values = np.array([burstparams[key][:N_inf] for key in inf_params]).flatten()

colors=['blue', 'red']
labels=['FM', 'MCMC']

In [ ]:
var_names = gen_parameter_labels(inf_params, N)
var_names = np.array(var_names).reshape(len(inf_params), N)[:, :N_inf].flatten()

fig = None
plt.rc('xtick',labelsize=18)
plt.rc('ytick',labelsize=18)

t0_labels = [f'$t_0^{i+1}$' for i in range(N_inf)]
log_r_labels = [f'$\log(r_{i+1})$' for i in range(N_inf)]
s_labels = [f'$s_{i+1}$' for i in range(N_inf)]
log_A_labels = [f'$\log(A_{i+1})$' for i in range(N_inf)]

for i in range(len(data)):
    fig = corner(
        data[i], 
        # labels=var_names,#['$t_0$', '$r$', '$s$', '$A$'], 
        labels=t0_labels + log_r_labels + s_labels + log_A_labels, 
        # truths=true_values, 
        # truth_color='darkslategrey', 
        range=[range_[i] for _ in range(N_inf * len(inf_params))], 
        color=colors[i], 
        label=labels[i], 
        fig=fig, 
        plot_density=True, 
        plot_datapoints=False, 
        fill_contours=False, plot_contours=True, 
        hist_kwargs={'density':True},
        # bins=50, 
        levels=[0.864, 0.393, 0.675], # [0.118, 0.393, 0.675, and 0.864] = []0.5, 1, 1.5, 2]-sigma
        smooth=1.5,
        max_n_ticks=3,
        labelpad=0.1,
        # use_math_text=True,
        # top_ticks=True,
        label_kwargs={'fontsize':26}
        )


# make legend with dummy lines
plt.legend(
        handles=[
            mlines.Line2D([], [], color=colors[i], label=labels[i])
            for i in range(len(data))
        ],
        fontsize=38, frameon=False,
        bbox_to_anchor=(1, 4), loc="upper right"
    )

# plt.savefig('FRB_115B_mcmc_fm_corner.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# save corner plot data
np.save('FM_samples_corner_plot_1_peak_15102039.npy', data[0])

# Posterior samples

In [ ]:
# posterior samples conditioned on N=N_inf
N_inf = 3
# FM_samples = final_snapshot[(Ns == N_inf).view(Ns.size(0))].cpu()
log_data = [FM_samples.cpu().numpy(), samples]

burstparams_inf = {key:np.array(value[:N_inf]) for key, value in burstparams.items()}
print(burstparams_inf, N_inf)
modelparams = {'time':np.linspace(0, 1, 1000), 'burstparams':burstparams_inf, 'ybkg':YBKG,'ncomp':N_inf}

true_flux = Model(**modelparams).get_flux()
# true_flux=None
plt.figure(figsize=(12,5))
from copy import deepcopy
def update_modelparams(sample, inf_params, modelparams):
    """
    Extract parameters from flat sample to update parameters dictionary.
    """
    N = modelparams['ncomp']

    for i, key in enumerate(inf_params):
        start = i * N
        stop  = start + N
        modelparams['burstparams'][key] = sample[start:stop]
    return modelparams

def plot_posterior_samples(N, simulated_counts, samples, inf_params, modelparams, true_flux=None, show=False, save_path=None, title=None, **kwargs):
    # plot the data
    time = modelparams["time"]
    plt.plot(time, simulated_counts, 'k-', alpha=0.3, label="data")
    
    # prevents changing the original modelparams
    modelparams_copy = deepcopy(modelparams)
    # print(modelparams)
    for j in range(N, 2*N):

        # draw random sample from posterior
        random_index = np.random.randint(low=0, high=len(samples))
        random_sample = samples[random_index]
        # generate noise-free curve from random sample
        modelparams_copy = update_modelparams(random_sample, inf_params, modelparams_copy)

        model = Model(**modelparams_copy).get_flux()

        # plot the sample
        plt.plot(time, model, alpha=0.2, label='posterior samples' if j == 0 else '', **kwargs)

    if true_flux is not None:
        plt.plot(np.linspace(0, 1, len(true_flux)), true_flux, '--', color='lightgrey', linewidth=1.2, label="ground-truth")
    
    plt.title(f'{N} posterior samples' if not title else title)
    plt.legend()
    plt.tight_layout()


for i in range(2):

    plt.subplot(121 + i)

    plot_posterior_samples(
        100, 
        simulated_counts, 
        log_data[i], 
        inf_params, 
        modelparams={'time':np.linspace(0, 1, 1000), 'burstparams':burstparams_inf, 'ybkg':YBKG,'ncomp':N_inf},
        true_flux=None, #true_flux,
        title=f"100 {labels[i]} posterior samples",
        color=colors[i],
        )
    # plt.title(None)
    plt.ylabel('counts', fontsize=14) if i == 0 else None
    # plt.ylim(0, 120)
    plt.xlim(0, 1)
    plt.grid(linestyle='dotted')
    plt.xlabel('t', fontsize=14)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    # if i == 1:
        # plt.yticks(ticks=[0, 20, 40, 60, 80, 100, 120], labels=[])

# plt.savefig('sample_curves_5_peaks_mcmc_fm.png', dpi=300)
plt.show()

In [ ]:
plt.savefig('sample_curves_5_peaks_mcmc_fm.png', dpi=300)
plt.show()

In [ ]:
# posterior samples that include all sampled component numbers
N = 1
BURSTPARAMS = {
        't0'   : torch.linspace(0.2, 0.8, N),
        'amp'  : torch.Tensor([np.log10(100)]).repeat(N),
        'rise' : torch.Tensor([np.log10(0.003)]).repeat(N),
        'skew' : torch.Tensor([5]).repeat(N)
    }

modelparams = {'time':torch.linspace(0, 1, 1000), 'burstparams': burstparams, 'ybkg':YBKG,'ncomp':1}

# true_flux = Model(**modelparams).get_flux()
# simulated_counts = torch.poisson(torch.tensor(true_flux, device=device))

plt.plot(simulated_counts, 'k', alpha=0.2)

posterior_param_samples = path.p_data.prior.samples_as_dict(theta_samples)
posterior_curve_samples = BatchedModel(device=device, **{'time':torch.linspace(0, 1, 1000), 'burstparams':posterior_param_samples, 'ybkg':YBKG,'ncomp':Ns}).get_flux()
for i in range(100):
    plt.plot(posterior_curve_samples[i].cpu(), alpha=0.7)

# plt.plot(true_flux)

# C2ST

Performs classifier two-sample test. A binary classifier is trained to distinguish samples from MCMC and FM, and its accuracy is evaluated. If the accuracy is 50\%, this indicates the classifier is unable to distinguish between the two distributions. If the accuracy is 100\%, the classifier can easily distinguish between samples from MCMC and FM. I.e. lower score is better and the 'best' minimum score is 50%. 

In [ ]:
# ensure number of MCMC samples equal to FM
MCMC_samples = np.copy(samples)
np.random.shuffle(MCMC_samples)
sample_size, _ = FM_samples.shape
MCMC_samples = torch.tensor(MCMC_samples[:sample_size], device=device, dtype=torch.float32)
FM_samples = torch.tensor(FM_samples, device=device, dtype=torch.float32)

# standardize using common mean and std
concat = torch.cat((FM_samples, MCMC_samples))
mean  = torch.mean(concat)
std  = torch.std(concat)

MCMC_samples_s = (MCMC_samples - mean) / std 
FM_samples_s   = (FM_samples - mean) / std 

accuracy = c2st(FM_samples_s, MCMC_samples_s, num_epochs=1000)

In [ ]:
print(f'c2st score: {accuracy}')

In [ ]:
FM_samples.shape

In [ ]:
MCMC_samples